# 01. Session Routing

`arcgateway` is the long-running daemon that makes ArcAgents reachable from
chat platforms — Telegram, Slack, Discord, web, email — without the agents
themselves caring about transports. This notebook walks the *core* of that
daemon: the `GatewayRunner` supervisor, the `SessionRouter` that funnels
inbound messages to per-(user, agent) sessions, and the `AsyncioExecutor`
that runs the agent turn in-process.

**You will learn:**

- Why a gateway sits between platforms and agents (multi-platform reachability).
- How `InboundEvent`, `Delta`, and `DeliveryTarget` form the routing contract.
- How `build_session_key` produces a stable per-(agent, user) identifier.
- The single most important concurrency bug in gateway code — the *pre-await
  race* — and how `SessionRouter` synchronously closes the window.
- How `AsyncioExecutor` runs an agent turn and yields streaming `Delta`s.
- How `GatewayRunner` supervises adapters with `asyncio.TaskGroup` so a
  crash in one platform never kills the others.
- How a synthetic platform adapter wires up to the runner end-to-end.

Everything runs without any platform credentials. The only moving parts are
in-memory: a fake "platform" emits `InboundEvent`s into the router, and we
observe routing, queueing, and delivery directly.

## 1. Setup

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

The boilerplate above puts every package's `src/` on `sys.path`, so we can
import `arcgateway` directly from the repo without an editable install. The
notebook is mock-first — no API keys required, no platform credentials,
no real network.

In [2]:
import arcgateway

print("arcgateway version:", arcgateway.__version__)
print("public API:", sorted(arcgateway.__all__))

arcgateway version: 0.2.0
public API: ['AsyncioExecutor', 'DeliveryTarget', 'Delta', 'Executor', 'GatewayRunner', 'InboundEvent', 'SessionRouter', 'build_session_key']


## 2. Why arcgateway — multi-platform agent reachability

Imagine you've built an `ArcAgent`. It can answer questions, call tools, and
remember conversation history. Now your team wants to talk to it from
Telegram. *And* Slack. *And* a web chat. *And* email. *And* keep the same
conversation thread when a user switches between Telegram and Slack.

You **don't** want each platform's quirks (Slack's `app_mention` events,
Telegram's `update_id` pagination, Discord's interaction tokens) leaking into
agent code. The agent should care about *messages*, not *transports*.

`arcgateway` is the layer that solves this. It:

1. Runs as a long-lived daemon (one process per deployment).
2. Owns the *adapters* that talk to each platform.
3. Normalises every inbound message into a single shape: `InboundEvent`.
4. Routes each event to a per-(user, agent) **session** — same user → same
   conversation, regardless of which platform they messaged from.
5. Streams agent output back as `Delta` chunks via a `DeliveryTarget`.

The four-pillar discipline (ADR-019) is enforced inside the gateway: every
inbound event is identified, signed/verified, authorised, and audited
*before* it reaches the agent. That story is covered in the next notebook
(`02-platform-adapters.ipynb`); this one focuses on the routing skeleton.

In [3]:
# Show the public surface so we know what we're working with.
from arcgateway import (
    AsyncioExecutor,
    DeliveryTarget,
    Delta,
    Executor,
    GatewayRunner,
    InboundEvent,
    SessionRouter,
    build_session_key,
)

for sym in [
    AsyncioExecutor,
    DeliveryTarget,
    Delta,
    Executor,
    GatewayRunner,
    InboundEvent,
    SessionRouter,
    build_session_key,
]:
    print(f"{sym.__name__:>16}  {type(sym).__name__}")

 AsyncioExecutor  type
  DeliveryTarget  ModelMetaclass
           Delta  ModelMetaclass
        Executor  _ProtocolMeta
   GatewayRunner  type
    InboundEvent  ModelMetaclass
   SessionRouter  type
build_session_key  function


Three of those are *data classes* (`InboundEvent`, `Delta`,
`DeliveryTarget`), two are *runtime objects* (`SessionRouter`,
`GatewayRunner`), and one is the *executor protocol* (`Executor`) plus its
in-process implementation (`AsyncioExecutor`). `build_session_key` is a
pure function. We'll exercise each in turn.

## 3. Core types — `InboundEvent`, `Delta`, `DeliveryTarget`

Every routing decision in `arcgateway` is made on these three Pydantic models.
Understanding their shape is understanding the gateway.

### 3.1 `InboundEvent` — what the adapter hands the router

When a platform adapter receives a message ("Telegram update", "Slack event",
"web POST"), it does the platform-specific decoding, then constructs an
`InboundEvent` and calls `session_router.handle(event)`. By that point all
platform quirks are gone: identity is resolved, the agent target is known,
and the session key is pre-computed.

In [4]:
from arcgateway import InboundEvent, build_session_key

agent_did = "did:arc:agent:assistant"
user_did = "did:arc:user:alice"

event = InboundEvent(
    platform="telegram",
    chat_id="12345",
    thread_id=None,
    user_did=user_did,
    agent_did=agent_did,
    session_key=build_session_key(agent_did, user_did),
    message="What is 17 squared?",
    raw_payload={"telegram_update_id": 998877},
)

print("platform    :", event.platform)
print("chat_id     :", event.chat_id)
print("user_did    :", event.user_did)
print("agent_did   :", event.agent_did)
print("session_key :", event.session_key)
print("message     :", event.message)
print("raw_payload :", event.raw_payload)

platform    : telegram
chat_id     : 12345
user_did    : did:arc:user:alice
agent_did   : did:arc:agent:assistant
session_key : 92ba475a97ed0557
message     : What is 17 squared?
raw_payload : {'telegram_update_id': 998877}


`raw_payload` is the full platform-specific blob, kept around for
audit and replay. The router itself never reads it — only the adapter and
the audit sink touch it. That separation is what lets the router stay
platform-agnostic.

### 3.2 `Delta` — what the executor streams back

When an agent turn runs, the executor doesn't return a single string. It
yields a sequence of `Delta` chunks: token text as it streams, tool-call
descriptions, and a final sentinel. The adapter consumes those chunks and
delivers them to the platform — typically by editing a single message in
place rather than spamming the channel.

In [5]:
from arcgateway import Delta

# A token chunk — partial agent output
token_delta = Delta(
    kind="token",
    content="The answer is ",
    is_final=False,
    turn_id="turn-001",
)

# A tool-call chunk — agent is invoking a tool
tool_delta = Delta(
    kind="tool_call",
    content="calculator(expression='17 ** 2')",
    is_final=False,
    turn_id="turn-001",
)

# The terminal sentinel — always the last delta of a turn
done_delta = Delta(
    kind="done",
    content="",
    is_final=True,
    turn_id="turn-001",
)

for d in (token_delta, tool_delta, done_delta):
    print(f"{d.kind:>10} | is_final={d.is_final} | content={d.content!r}")

     token | is_final=False | content='The answer is '
 tool_call | is_final=False | content="calculator(expression='17 ** 2')"
      done | is_final=True | content=''


Three `kind` values, exactly. `token` for streamed text, `tool_call`
for tool invocations, `done` for the terminal sentinel. Every well-behaved
executor must yield a `Delta(kind='done', is_final=True)` as its last
output — that's the contract. `StreamBridge` (covered in notebook 02)
relies on it to know when to flush the final platform message.

### 3.3 `DeliveryTarget` — the address for outbound messages

Outbound delivery is symmetric: the adapter receives an `InboundEvent`, and
when it's time to reply, the gateway hands it a `DeliveryTarget`. The target
is a parsed `platform:chat_id` (or `platform:chat_id:thread_id`) tuple.
Same address format, same colon-delimited round-trippable string — works
in TOML config, CLI args, audit logs, anywhere.

In [6]:
from arcgateway import DeliveryTarget

# Bare platform:chat_id
t1 = DeliveryTarget.parse("telegram:12345")
print("t1:", t1, "thread_id =", t1.thread_id)

# With thread_id (Slack thread_ts, Telegram message thread, Discord thread)
t2 = DeliveryTarget.parse("slack:C123ABC:T9876")
print("t2:", t2, "thread_id =", t2.thread_id)

# Round-trips cleanly
assert str(t1) == "telegram:12345"
assert str(t2) == "slack:C123ABC:T9876"
print("round-trip ok")

t1: telegram:12345 thread_id = None
t2: slack:C123ABC:T9876 thread_id = T9876
round-trip ok


The `parse()` shape is intentionally permissive: at most three
colon-delimited parts. Anything fewer than two raises `ValueError`. The
platform name is normalised to lowercase, and unknown platforms get a
warning (not an error — adapters can register at runtime).

In [7]:
# Empty chat_id is rejected — every platform needs an explicit target.
try:
    DeliveryTarget(platform="slack", chat_id="")
except Exception as exc:
    print(f"Rejected empty chat_id: {type(exc).__name__}")

# Malformed string raises ValueError
try:
    DeliveryTarget.parse("just-one-token")
except ValueError as exc:
    print("ValueError:", exc)

Rejected empty chat_id: ValidationError
ValueError: Invalid DeliveryTarget format 'just-one-token'. Expected 'platform:chat_id' or 'platform:chat_id:thread_id'.


The known-platform list is a soft check, not a gate — `_KNOWN_PLATFORMS`
in `delivery.py` covers the common cases (`telegram`, `slack`, `discord`,
`whatsapp`, `signal`, `matrix`, `email`, `web`), and unknown platforms log
a warning. This lets you register a custom adapter without patching the
core.

In [8]:
# Unknown platform → warning, but the object is still constructed.
import logging

logging.basicConfig(level=logging.WARNING)
custom = DeliveryTarget(platform="my-custom-bridge", chat_id="room-42")
print("custom target:", custom)

custom target: my-custom-bridge:room-42


## 4. `SessionRouter` and `build_session_key`

The session is the unit of conversational state. One user talking to one
agent is one session. The session **key** is what the router uses as a dict
key to track in-flight turns and queued events.

### 4.1 `build_session_key` — deterministic from (agent, user)

The key is a truncated SHA-256 of `f"{agent_did}:{user_did}"`. Sixteen hex
characters — `2^64` preimage resistance, plenty for in-memory uniqueness
while staying readable in logs. **Crucially, the platform is not part of
the key.** That's what enables cross-platform session continuity (D-06):
the same user messaging the same agent from Telegram, then Slack, lands in
the same session — provided the identity-graph layer resolved both
platform IDs to the same `user_did`.

In [9]:
from arcgateway import build_session_key

agent = "did:arc:agent:assistant"
alice = "did:arc:user:alice"
bob = "did:arc:user:bob"

# Deterministic — same inputs, same key
key1 = build_session_key(agent, alice)
key2 = build_session_key(agent, alice)
assert key1 == key2
print(f"alice + assistant: {key1}")

# Different user → different key
key_bob = build_session_key(agent, bob)
assert key_bob != key1
print(f"bob   + assistant: {key_bob}")

# Different agent → different key (same user, two agents = two sessions)
key_coder = build_session_key("did:arc:agent:coder", alice)
assert key_coder != key1
print(f"alice + coder    : {key_coder}")

# All keys are 16 lowercase hex chars
for k in (key1, key_bob, key_coder):
    assert len(k) == 16
    assert all(c in "0123456789abcdef" for c in k)
print("\nall keys: 16 hex chars, lowercase — ok")

alice + assistant: 92ba475a97ed0557
bob   + assistant: 156e96c9e674813e
alice + coder    : 3c7f971ba37efc9e

all keys: 16 hex chars, lowercase — ok


### 4.2 Cross-platform continuity

The platform name is *deliberately omitted* from `build_session_key`. If
Alice messages the assistant from Telegram, then later from Slack, both
inbound events resolve to the same `user_did` (via the identity graph
layer) and therefore the same session key — giving her one continuous
conversation.

In [10]:
# Same user, same agent, two platforms — same key.
key_via_telegram = build_session_key(agent, alice)
key_via_slack = build_session_key(agent, alice)
assert key_via_telegram == key_via_slack
print("cross-platform same-user same-agent → identical session key")
print(f"  telegram: {key_via_telegram}")
print(f"  slack   : {key_via_slack}")

cross-platform same-user same-agent → identical session key
  telegram: 92ba475a97ed0557
  slack   : 92ba475a97ed0557


### 4.3 Building a `SessionRouter`

A router needs an `Executor`. The executor runs the agent turn and yields
`Delta`s. For testing and notebooks we use an in-memory fake — same
contract, no LLM calls.

In [11]:
from collections.abc import AsyncIterator

from arcgateway import Delta, InboundEvent


class EchoExecutor:
    """Minimal Executor implementation: echoes the message back as one delta."""

    async def run(self, event: InboundEvent) -> AsyncIterator[Delta]:
        return self._stream(event)

    async def _stream(self, event: InboundEvent) -> AsyncIterator[Delta]:
        yield Delta(
            kind="token",
            content=f"echo: {event.message}",
            is_final=False,
            turn_id=event.session_key,
        )
        yield Delta(kind="done", content="", is_final=True, turn_id=event.session_key)


# It satisfies the Protocol because it has a matching `run` coroutine.
from arcgateway import Executor

assert isinstance(EchoExecutor(), Executor)
print("EchoExecutor satisfies the Executor Protocol")

EchoExecutor satisfies the Executor Protocol


In [12]:
from arcgateway import SessionRouter

router = SessionRouter(executor=EchoExecutor())
print("router         :", router)
print("active sessions:", router.active_session_count())
print("queue (empty)  :", router.queue_depth("nonexistent-key"))

router         : <arcgateway.session.SessionRouter object at 0x10cf0ec90>
active sessions: 0
queue (empty)  : 0


### 4.4 First inbound event

We hand-build an `InboundEvent` (in production the adapter does this) and
call `router.handle(event)`. That spawns an asyncio task to run the turn —
`handle` itself returns immediately.

In [13]:
import asyncio

agent_did = "did:arc:agent:assistant"
user_did = "did:arc:user:alice"
session_key = build_session_key(agent_did, user_did)

event = InboundEvent(
    platform="telegram",
    chat_id="12345",
    user_did=user_did,
    agent_did=agent_did,
    session_key=session_key,
    message="hello",
)

await router.handle(event)
# Yield once so the spawned task gets a chance to start.
await asyncio.sleep(0)

print("agent_tasks_spawned:", router.agent_tasks_spawned)
print("active sessions   :", router.active_session_count())

agent_tasks_spawned: {'92ba475a97ed0557': 1}
active sessions   : 0


The test-instrumentation counters (`agent_tasks_spawned`,
`queued_events`) are exposed for tests; production code never branches on
them. `active_session_count()` is the legitimate observability hook.

In [14]:
# Wait briefly for the echo to finish so the test counters settle.
await asyncio.sleep(0.05)
print("after settle:")
print("  active sessions   :", router.active_session_count())
print("  agent_tasks_spawned:", router.agent_tasks_spawned)

after settle:
  active sessions   : 0
  agent_tasks_spawned: {'92ba475a97ed0557': 1}


## 5. The race-condition guard — most important code in the file

This is the single most dangerous concurrency bug in gateway implementations,
and the comment block at the top of `session.py` exists for it.

**The bug (Hermes PR #4926):** Two messages from the same user arrive in the
same event-loop tick. Without a synchronous guard, both coroutines pass the
`if session_key in self._active_sessions` check before either has had a
chance to insert into the dict. Two competing agent tasks spawn for the same
session — double responses, interleaved LLM context, audit log corruption.

**The fix:** Insert into `_active_sessions` *synchronously*, before any
`await`, in the same event-loop tick as the guard check. Python's asyncio
guarantees no other coroutine runs between two synchronous statements within
the same task. Only an explicit `await` is a preemption point.

### 5.1 The wrong shape (don't do this)

```python
# WRONG — race window between check and assignment
if session_key not in self._active_sessions:
    await something()                          # ← preemption point!
    self._active_sessions[session_key] = asyncio.Event()
```

### 5.2 The right shape (what `session.py` actually does)

```python
# RIGHT — no await between check and assignment
if session_key in self._active_sessions:
    self._queue_for_session(session_key, event)
    return
self._active_sessions[session_key] = asyncio.Event()  # SYNC — no await above
asyncio.create_task(self._process_session(...))
```

Two synchronous statements. No `await` between them. The race window is
exactly zero ticks wide.

### 5.3 Demonstrating the guard with 20 concurrent events

We fire 20 `InboundEvent`s with the **same** session key into the router
via `asyncio.gather`. They all arrive in the same event-loop iteration. The
guard must spawn exactly **one** agent task and queue the other 19.

In [15]:
class GatedExecutor:
    """Holds turns open until released — keeps the session 'busy'."""

    def __init__(self) -> None:
        self._gates: dict[str, asyncio.Event] = {}

    def release(self, session_key: str) -> None:
        gate = self._gates.setdefault(session_key, asyncio.Event())
        gate.set()

    def release_all(self) -> None:
        for gate in self._gates.values():
            gate.set()

    async def run(self, event: InboundEvent) -> AsyncIterator[Delta]:
        # Pre-create the gate so release() works before _stream runs.
        self._gates.setdefault(event.session_key, asyncio.Event())
        return self._stream(event)

    async def _stream(self, event: InboundEvent) -> AsyncIterator[Delta]:
        gate = self._gates[event.session_key]
        await gate.wait()
        yield Delta(kind="done", content="", is_final=True, turn_id=event.session_key)


executor = GatedExecutor()
race_router = SessionRouter(executor=executor)

agent_did = "did:arc:agent:bot"
user_did = "did:arc:user:alice"
session_key = build_session_key(agent_did, user_did)

events = [
    InboundEvent(
        platform="telegram",
        chat_id="12345",
        user_did=user_did,
        agent_did=agent_did,
        session_key=session_key,
        message=f"concurrent #{i}",
    )
    for i in range(20)
]

# Fire 20 handle() calls concurrently — same session key.
await asyncio.gather(*[race_router.handle(e) for e in events])
await asyncio.sleep(0)

spawned = race_router.agent_tasks_spawned[session_key]
queued = len(race_router.queued_events[session_key])
print(f"tasks spawned: {spawned}  (must be 1 — guard worked)")
print(f"events queued: {queued}  (remaining 19, none dropped)")

assert spawned == 1
assert queued == 19

tasks spawned: 1  (must be 1 — guard worked)
events queued: 19  (remaining 19, none dropped)


One task. Nineteen queued. No drops. The guard works because there is
no `await` between the check and the assignment in `SessionRouter.handle`.

That single property is enforced by an integration test
(`tests/integration/test_race_regression.py`) that runs at concurrency
levels 5, 10, 20, 50 — *and* a separate stress test
(`test_race_regression_stress.py`) that runs at higher counts. If you ever
modify `handle()`, that test is the canary.

In [16]:
# Drain the queue cleanly so we don't leave dangling tasks.
executor.release_all()
await asyncio.sleep(0.2)

print("after drain:")
print("  tasks spawned:", race_router.agent_tasks_spawned[session_key])
print("  active       :", race_router.active_session_count())
print("  queue depth  :", race_router.queue_depth(session_key))

after drain:
  tasks spawned: 20
  active       : 0
  queue depth  : 0


All 20 messages eventually run — sequentially, one turn at a time —
through the queue-drain path in `_process_session` / `_drain_queue`. The
guard prevents *concurrent* turns for the same session; it never drops
messages.

## 6. `AsyncioExecutor` — the in-process tier

Personal and enterprise tiers use `AsyncioExecutor`: the agent runs in the
same process, in the same event loop as the gateway. Cheap to start (no
subprocess fork), shared connection pools, but no OS-level isolation.

Federal tier swaps in `SubprocessExecutor` for OS-level isolation —
each session in its own subprocess with resource limits — for CMMC/FedRAMP
compliance. Same `Executor` Protocol, different stringency. **Tier is a
stringency knob, not a pillar bypass.** Both tiers still identify, sign,
authorise, and audit (ADR-019).

### 6.1 The echo stub (no `agent_factory`)

When `AsyncioExecutor` is constructed without an `agent_factory`, it uses
a built-in echo stub. This is what powers tests and notebook demos that
don't want to wire up a real ArcAgent.

In [17]:
from arcgateway import AsyncioExecutor

stub_executor = AsyncioExecutor()

stub_event = InboundEvent(
    platform="web",
    chat_id="browser-tab-1",
    user_did="did:arc:user:demo",
    agent_did="did:arc:agent:assistant",
    session_key=build_session_key("did:arc:agent:assistant", "did:arc:user:demo"),
    message="ping",
)

stream = await stub_executor.run(stub_event)
async for delta in stream:
    print(f"{delta.kind:>6} | final={delta.is_final} | {delta.content!r}")

 token | final=False | "[AsyncioExecutor stub] Received: 'ping' (session=54d13f60f5f9bbda)"
  done | final=True | ''


Two deltas: one `token` containing the echo response, one `done`
sentinel. That's the minimum contract every executor must satisfy.

### 6.2 Wiring an `agent_factory`

In production, `AsyncioExecutor` is constructed with an async
`agent_factory(agent_did) -> agent` callable. The executor calls
`await agent.chat(message, session_id=session_key)` (or `agent.run` if
`chat` is absent), wraps the result in `Delta`s, and streams them back.

In [18]:
class FakeAgent:
    """Stand-in for ArcAgent with a chat() method."""

    def __init__(self, agent_did: str) -> None:
        self.agent_did = agent_did
        self.calls: list[tuple[str, str]] = []

    async def chat(self, message: str, *, session_id: str) -> object:
        self.calls.append((message, session_id))
        # ArcAgent.chat returns a result with a `.content` attribute.
        return type("Result", (), {"content": f"[{self.agent_did}] handled: {message!r}"})()


# Per-DID agent cache — the factory is called once per event but a real
# implementation would cache (this fake doesn't bother).
async def make_agent(agent_did: str) -> FakeAgent:
    return FakeAgent(agent_did)


real_executor = AsyncioExecutor(agent_factory=make_agent)

stream = await real_executor.run(stub_event)
async for delta in stream:
    print(f"{delta.kind:>6} | final={delta.is_final} | {delta.content!r}")

ERROR:arcgateway.executor:AsyncioExecutor: agent error session=54d13f60f5f9bbda: 'FakeAgent' object has no attribute 'session'
Traceback (most recent call last):
  File "/Users/joshschultz/Projects/arc/.claude/worktrees/agent-a7b1b963ad73e08f9/packages/arcgateway/src/arcgateway/executor.py", line 255, in _stream
    session = await agent.session(event.session_key)
                    ^^^^^^^^^^^^^
AttributeError: 'FakeAgent' object has no attribute 'session'


 token | final=False | '[agent-error] the run failed; see server logs'
  done | final=True | ''


The `set_agent_factory` and `agent_factory` property are the
extension points used by `arcui.embedded_agents.install_embedded_agent_hooks`
to *wrap* the bootstrap-built factory with caching and a fleet-registry
hook — atomic replace, no private-attribute mutation.

In [19]:
# Atomic replace of the agent factory at runtime.
captured: list[str] = []


async def wrapping_factory(agent_did: str) -> FakeAgent:
    captured.append(agent_did)
    return FakeAgent(agent_did)


real_executor.set_agent_factory(wrapping_factory)

# Run another turn to prove the wrapped factory is now in effect.
stream = await real_executor.run(stub_event)
async for delta in stream:
    if delta.kind == "token":
        print("token:", delta.content)

print("captured agent DIDs:", captured)
print("current factory    :", real_executor.agent_factory.__name__)

ERROR:arcgateway.executor:AsyncioExecutor: agent error session=54d13f60f5f9bbda: 'FakeAgent' object has no attribute 'session'
Traceback (most recent call last):
  File "/Users/joshschultz/Projects/arc/.claude/worktrees/agent-a7b1b963ad73e08f9/packages/arcgateway/src/arcgateway/executor.py", line 255, in _stream
    session = await agent.session(event.session_key)
                    ^^^^^^^^^^^^^
AttributeError: 'FakeAgent' object has no attribute 'session'


token: [agent-error] the run failed; see server logs
captured agent DIDs: ['did:arc:agent:assistant']
current factory    : wrapping_factory


### 6.3 Errors are caught and turned into deltas

If the agent raises, `AsyncioExecutor` catches the exception, logs it
structured, and yields a `[agent-error]` token followed by the `done`
sentinel — the contract is preserved. The session never hangs.

In [20]:
class ExplodingAgent:
    async def chat(self, message: str, *, session_id: str) -> object:
        raise RuntimeError(f"boom on {message!r}")


async def exploding_factory(agent_did: str) -> ExplodingAgent:
    return ExplodingAgent()


error_executor = AsyncioExecutor(agent_factory=exploding_factory)

stream = await error_executor.run(stub_event)
async for delta in stream:
    print(f"{delta.kind:>6} | final={delta.is_final} | {delta.content!r}")

ERROR:arcgateway.executor:AsyncioExecutor: agent error session=54d13f60f5f9bbda: 'ExplodingAgent' object has no attribute 'session'
Traceback (most recent call last):
  File "/Users/joshschultz/Projects/arc/.claude/worktrees/agent-a7b1b963ad73e08f9/packages/arcgateway/src/arcgateway/executor.py", line 255, in _stream
    session = await agent.session(event.session_key)
                    ^^^^^^^^^^^^^
AttributeError: 'ExplodingAgent' object has no attribute 'session'


 token | final=False | '[agent-error] the run failed; see server logs'
  done | final=True | ''


The error is wrapped, the `done` sentinel is still emitted, the
session lifecycle finishes cleanly. That's the kind of failure containment
ASI08 demands — a crashed agent never cascades into a stuck session.

## 7. `GatewayRunner` — supervising platform adapters

The runner is the entry point for `asyncio.run()` in the daemon. Its job:

1. Build a `SessionRouter` (one per gateway).
2. Start each registered adapter inside an `asyncio.TaskGroup`, so a
   crash in one adapter never kills the others (ASI08 — cascading-failure
   containment).
3. Run a *reconnect watcher* alongside, restarting failed adapters with
   exponential backoff.
4. Install SIGINT/SIGTERM handlers for clean shutdown.
5. Write a PID file (`gateway.pid`) atomically at startup, refusing to
   start if a live instance already holds it.
6. Write a `.clean_shutdown` marker on graceful exit so supervisors can
   distinguish clean stop from crash restart.

We won't run the full daemon (it'd block on `asyncio.run` waiting for
SIGTERM), but we'll wire one up and inspect every accessible piece.

### 7.1 Constructing a runner

The constructor takes optional `adapters`, `executor`, and `runtime_dir`.
We pass an `AsyncioExecutor` and a temp dir so we don't pollute the
default `~/.arc/gateway/run`.

In [21]:
import tempfile
from pathlib import Path

from arcgateway import GatewayRunner

tmp = Path(tempfile.mkdtemp(prefix="arcgw_nb_"))
print("runtime_dir:", tmp)

runner = GatewayRunner(
    adapters=[],  # we'll add adapters below
    executor=AsyncioExecutor(),
    runtime_dir=tmp,
)
print("runner          :", runner)
print("session_router  :", runner.session_router)


runtime_dir: /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arcgw_nb_34uhh2hc
runner          : <arcgateway.runner.GatewayRunner object at 0x10c1fe330>
session_router  : <arcgateway.session.SessionRouter object at 0x10c1ed550>


### 7.2 Outbound sends go through adapters

`GatewayRunner` keeps an adapter index keyed by platform name. Streaming
responses are delivered by the `StreamBridge` calling `adapter.send()` per
turn. Scheduled / proactive delivery ("agent says something at 9am") is
owned by the agent-side platform modules, which consume scheduler bus
events and send through their bots — the gateway carries no separate
delivery-sender component.

In [22]:
print("adapters indexed:", list(runner._adapter_index))  # empty until add_adapter below

adapters indexed: []


### 7.3 A synthetic platform adapter

We won't import the real `BasePlatformAdapter` (that's covered in notebook
02). For now, here's a minimal stand-in that exposes:

- `name` — used as the dict key in the delivery sender.
- `connect()` / `disconnect()` — adapter lifecycle.
- `send(target, message)` — outbound delivery.
- `on_message` — closure over the router's `handle`.

In production, `connect()` opens a long-poll/WebSocket to the platform and
keeps it alive. Here `connect()` returns immediately and the adapter waits
on a synthetic event we'll fire by hand.

In [23]:
class SyntheticAdapter:
    """Pretend platform that fires synthetic InboundEvents into a router."""

    def __init__(self, name: str) -> None:
        self.name = name
        self.connected = False
        self.delivered: list[tuple[DeliveryTarget, str]] = []
        self._router: SessionRouter | None = None

    def bind(self, router: SessionRouter) -> None:
        self._router = router

    async def connect(self) -> None:
        self.connected = True
        print(f"  [{self.name}] connect()")

    async def disconnect(self) -> None:
        self.connected = False
        print(f"  [{self.name}] disconnect()")

    async def send(self, target: DeliveryTarget, message: str) -> None:
        """Outbound — the gateway calls this to deliver agent replies."""
        self.delivered.append((target, message))
        print(f"  [{self.name}] send → {target} | {message!r}")

    async def emit(self, message: str, *, user_did: str, agent_did: str) -> None:
        """Simulate the platform pushing an inbound message into the router."""
        assert self._router is not None, "bind() before emit()"
        event = InboundEvent(
            platform=self.name,
            chat_id="synthetic-chat-1",
            user_did=user_did,
            agent_did=agent_did,
            session_key=build_session_key(agent_did, user_did),
            message=message,
        )
        await self._router.handle(event)

### 7.4 Registering adapters with the runner

`add_adapter` appends to `runner._adapters` and indexes the adapter by its
platform name, so inbound routing — and direct outbound sends — can find it.

In [24]:
telegram = SyntheticAdapter("telegram")
slack = SyntheticAdapter("slack")

runner.add_adapter(telegram)
runner.add_adapter(slack)

print("adapter index    :", list(runner._adapter_index))


adapter index    : ['telegram', 'slack']


Both adapters now appear in the delivery sender's routing table, so a
cron job that emits `slack:C123` will reach the Slack adapter and a job
emitting `telegram:99` will reach the Telegram adapter — the runner does
the wiring at startup.

## 8. End-to-end synthetic-platform demo

Now we wire it all together: bind the synthetic adapters to the router,
emit a few inbound events, and watch the routing happen. We won't call
`runner.run()` (it'd block), but everything *inside* `run()` after task
construction is exactly what we're exercising here.

In [25]:
# Bind the synthetic adapters to the runner's session router so emit()
# can hand events off.
telegram.bind(runner.session_router)
slack.bind(runner.session_router)

print("router :", runner.session_router)
print("ready  :", runner.session_router.active_session_count(), "active sessions")

router : <arcgateway.session.SessionRouter object at 0x10c1ed550>
ready  : 0 active sessions


### 8.1 Two users, one agent — independent sessions

Alice and Bob both message the assistant from Telegram. Because their
`user_did`s differ, `build_session_key` produces two different keys, and
two independent agent tasks run concurrently.

In [26]:
alice_did = "did:arc:user:alice"
bob_did = "did:arc:user:bob"
agent_did = "did:arc:agent:assistant"

await telegram.emit("hello from alice", user_did=alice_did, agent_did=agent_did)
await telegram.emit("hello from bob", user_did=bob_did, agent_did=agent_did)
await asyncio.sleep(0.05)

print("active sessions  :", runner.session_router.active_session_count())
print("tasks spawned    :", runner.session_router.agent_tasks_spawned)

  [telegram] send → telegram:synthetic-chat-1 | "[AsyncioExecutor stub] Received: 'hello from alice' (session=92ba475a97ed0557)"
  [telegram] send → telegram:synthetic-chat-1 | "[AsyncioExecutor stub] Received: 'hello from bob' (session=156e96c9e674813e)"
active sessions  : 0
tasks spawned    : {'92ba475a97ed0557': 1, '156e96c9e674813e': 1}


Two sessions, each with one task spawned. The keys are derived from
`(agent_did, user_did)` only — platform doesn't enter the equation.

### 8.2 Same user, two platforms — one session

Now Alice messages from Slack. Same `user_did`, same `agent_did` ⇒ same
session key. The router treats this as a continuation of Alice's existing
conversation, even though it arrived via a different adapter.

In [27]:
await slack.emit("now from slack!", user_did=alice_did, agent_did=agent_did)
await asyncio.sleep(0.05)

# Same session key for Alice across both platforms.
key_alice = build_session_key(agent_did, alice_did)
print(f"alice's session key (telegram + slack): {key_alice}")
print(
    f"tasks spawned for that key           : {runner.session_router.agent_tasks_spawned[key_alice]}"
)

  [slack] send → slack:synthetic-chat-1 | "[AsyncioExecutor stub] Received: 'now from slack!' (session=92ba475a97ed0557)"
alice's session key (telegram + slack): 92ba475a97ed0557
tasks spawned for that key           : 2


Two tasks have run for Alice's session — first her Telegram message,
then her Slack one. The conversation history (in a real `ArcAgent`)
threads through both. That's D-06 cross-platform continuity, made
mechanical by `build_session_key`.

### 8.3 Outbound delivery via `adapter.send`

The same adapter index that powered inbound events powers outbound
delivery: look the adapter up by the target's platform and call `send`.

In [28]:
target = DeliveryTarget(platform="telegram", chat_id="12345")
await runner._adapter_index[target.platform].send(target, "Daily summary ready.")

slack_target = DeliveryTarget.parse("slack:C123ABC")
await runner._adapter_index[slack_target.platform].send(slack_target, "Build #42 succeeded.")

# Each adapter recorded its outbound deliveries.
print("\ntelegram deliveries:")
for target, msg in telegram.delivered:
    print(f"  {target} → {msg!r}")
print("slack deliveries:")
for target, msg in slack.delivered:
    print(f"  {target} → {msg!r}")

  [telegram] send → telegram:12345 | 'Daily summary ready.'
  [slack] send → slack:C123ABC | 'Build #42 succeeded.'

telegram deliveries:
  telegram:synthetic-chat-1 → "[AsyncioExecutor stub] Received: 'hello from alice' (session=92ba475a97ed0557)"
  telegram:synthetic-chat-1 → "[AsyncioExecutor stub] Received: 'hello from bob' (session=156e96c9e674813e)"
  telegram:12345 → 'Daily summary ready.'
slack deliveries:
  slack:synthetic-chat-1 → "[AsyncioExecutor stub] Received: 'now from slack!' (session=92ba475a97ed0557)"
  slack:C123ABC → 'Build #42 succeeded.'


One sender, multiple platforms, no platform-specific code in the
caller. That's the routing skeleton.

## 9. Failure modes — what can go wrong

Knowing the happy path isn't enough. Here are the failure modes the
gateway has to handle, and how it does.

### 9.1 Two gateway instances on the same `runtime_dir`

The runner writes a PID file atomically at startup. If a live process
already holds it, startup raises `GatewayAlreadyRunning` — no second
instance can clobber the first. Stale PID files (process gone) are
overwritten with a warning.

In [29]:
from arcgateway.runner import GatewayAlreadyRunning, _pid_is_alive

# Simulate a live PID file by writing the current process's PID.
fake_runtime = Path(tempfile.mkdtemp(prefix="arcgw_pid_"))
pid_file = fake_runtime / "gateway.pid"
pid_file.write_text(f"{os.getpid()}\n")
print("PID file contents:", pid_file.read_text().strip())
print("PID alive?       :", _pid_is_alive(os.getpid()))

# Constructing a runner is fine; calling run() would raise.
guarded = GatewayRunner(adapters=[], executor=AsyncioExecutor(), runtime_dir=fake_runtime)

# Inspect the guard machinery directly.
try:
    guarded._write_pid_file()
except GatewayAlreadyRunning as exc:
    print("\nGatewayAlreadyRunning raised:")
    print(" ", exc)

PID file contents: 18154
PID alive?       : True

GatewayAlreadyRunning raised:
  arcgateway is already running (pid=18154, runtime_dir=/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arcgw_pid_cs50wofh). Stop it with `arc gateway stop` before starting a new instance.


### 9.2 Adapter crashes — sibling adapters survive

`GatewayRunner._run_adapter` catches every exception and writes the
adapter into `_failed_adapters` for the reconnect watcher to retry.
Crucially, the exception does **not** propagate to the `TaskGroup` —
that would cancel the sibling adapters.

In [30]:
class CrashingAdapter:
    name = "discord"

    async def connect(self) -> None:
        raise ConnectionError("auth failed")

    async def disconnect(self) -> None:
        pass

    async def send(self, target: DeliveryTarget, message: str) -> None:  # pragma: no cover
        raise NotImplementedError


crash = CrashingAdapter()

# Just call _run_adapter directly to show the exception is captured.
crash_runner = GatewayRunner(
    adapters=[crash], executor=AsyncioExecutor(), runtime_dir=Path(tempfile.mkdtemp())
)

await crash_runner._run_adapter(crash)

print("failed_adapters:", list(crash_runner._failed_adapters))
print("entry          :", crash_runner._failed_adapters["discord"])

ERROR:arcgateway.runner:Adapter discord failed: auth failed
Traceback (most recent call last):
  File "/Users/joshschultz/Projects/arc/.claude/worktrees/agent-a7b1b963ad73e08f9/packages/arcgateway/src/arcgateway/runner.py", line 355, in _run_adapter
    await adapter.connect()
  File "/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/ipykernel_18154/4199281343.py", line 5, in connect
    raise ConnectionError("auth failed")
ConnectionError: auth failed


failed_adapters: ['discord']
entry          : FailedAdapter(name='discord', attempt=0, last_error=ConnectionError('auth failed'), permanently_failed=False)


The crash is recorded, the runner is unharmed, and the reconnect
watcher (which we didn't start here) would pick it up on its next poll.
Sibling adapters (Telegram, Slack) keep running.

### 9.3 Unknown delivery platform

`DeliveryTarget` lower-cases platform names and logs (rather than raises)
on unknown ones — the target object is always constructible. The failure
happens one step later, at delivery time: `GatewayRunner._adapter_index`
is a plain `dict[str, BasePlatformAdapter]` keyed by registered adapter
name, so looking up a platform with no registered adapter raises
`KeyError`. There is no separate delivery-sender component that validates
platform routing — the index lookup *is* the gate.

In [31]:
discord_target = DeliveryTarget(platform="discord", chat_id="X")
try:
    await runner._adapter_index[discord_target.platform].send(discord_target, "hi")
except KeyError as exc:
    print("KeyError: no adapter registered for platform", exc)

KeyError: no adapter registered for platform 'discord'


### 9.4 Malformed `DeliveryTarget` strings

`DeliveryTarget.parse` rejects malformed inputs. Empty `chat_id` is
rejected at the field-validator level (every platform requires an explicit
target).

In [32]:
for bad in ("only-platform", "telegram::", "  :  ", ""):
    try:
        DeliveryTarget.parse(bad)
        print(f"  {bad!r} → unexpectedly parsed")
    except (ValueError, Exception) as exc:
        print(f"  {bad!r:30} → {type(exc).__name__}: {str(exc)[:80]}")

  'only-platform'                → ValueError: Invalid DeliveryTarget format 'only-platform'. Expected 'platform:chat_id' or 'p
  'telegram::'                   → ValidationError: 1 validation error for DeliveryTarget
chat_id
  Value error, chat_id must not be
  '  :  '                        → ValidationError: 1 validation error for DeliveryTarget
chat_id
  Value error, chat_id must not be
  ''                             → ValueError: Invalid DeliveryTarget format ''. Expected 'platform:chat_id' or 'platform:chat_


### 9.5 Executor that forgets the `done` sentinel

If an executor implementation never yields `Delta(kind='done', is_final=True)`,
the consumer (StreamBridge in production) will wait forever for the
terminal flush. The contract is "every turn ends with `done`" — anything
else is a bug. The `Executor` Protocol can't enforce this at the type
level, but the integration tests do.

In [33]:
class BrokenExecutor:
    async def run(self, event: InboundEvent) -> AsyncIterator[Delta]:
        return self._stream(event)

    async def _stream(self, event: InboundEvent) -> AsyncIterator[Delta]:
        # Forgets to emit the done sentinel — never do this in real code.
        yield Delta(kind="token", content="partial", is_final=False, turn_id="t1")


broken_router = SessionRouter(executor=BrokenExecutor())
await broken_router.handle(stub_event)
await asyncio.sleep(0.05)

# The session completes (the generator exhausts), but in production the
# StreamBridge would have nothing to flush as the final message — the user
# would see "partial" and no final state.
print("active sessions after broken turn:", broken_router.active_session_count())
print("(a real adapter would never see the 'done' it's waiting for)")

active sessions after broken turn: 0
(a real adapter would never see the 'done' it's waiting for)


### 9.6 Pre-await race regression — caught by integration test

We already exercised this in §5. The point worth repeating: the *only*
thing standing between a correct gateway and a quietly broken one is
the absence of `await` between the guard check and the dict assignment
in `SessionRouter.handle`. If you ever refactor that method, run
`test_race_regression.py` — it fires N=20 events at the same key and
asserts exactly one task spawned.

## 10. Summary

The gateway routing skeleton is small but every piece is load-bearing.

**What we covered:**

- **`InboundEvent`** — normalised platform-agnostic message; `raw_payload`
  preserves the platform-specific blob for audit/replay.
- **`Delta`** — three `kind`s only (`token`, `tool_call`, `done`); every
  turn ends with `Delta(kind='done', is_final=True)`.
- **`DeliveryTarget`** — colon-delimited `platform:chat_id[:thread_id]`,
  parsed by `DeliveryTarget.parse`; round-trips cleanly.
- **`build_session_key(agent_did, user_did)`** — deterministic 16-hex SHA-256
  truncation; platform is *deliberately* omitted, enabling cross-platform
  session continuity (D-06).
- **`SessionRouter`** — owns `_active_sessions`, queues concurrent events
  per session via `QueueManager`, surfaces `active_session_count()` and
  `queue_depth()` for observability.
- **The race-condition guard** — synchronous check-and-assign of
  `_active_sessions` with no `await` between them; integration test at
  N=20 enforces it.
- **`AsyncioExecutor`** — in-process tier; takes an optional
  `agent_factory(agent_did) -> agent`; supports `set_agent_factory` for
  atomic runtime replacement; errors become `[agent-error]` deltas, never
  hanging sessions.
- **`Executor` Protocol** — `runtime_checkable`, contract is
  "`run()` returns an `AsyncIterator[Delta]`, last delta is `done`".
- **`GatewayRunner`** — supervises adapters in an `asyncio.TaskGroup`
  (sibling crashes contained), writes `gateway.pid` atomically and
  refuses to start if a live instance holds it
  (`GatewayAlreadyRunning`), writes `.clean_shutdown` marker on graceful
  exit. `_adapter_index` is rebuilt automatically as adapters are added
  via `runner.add_adapter`; an unregistered platform raises `KeyError`
  at delivery time rather than silently dropping.

**Public API exercised** (matches `arcgateway.__all__`):
`AsyncioExecutor`, `DeliveryTarget`, `Delta`, `Executor`, `GatewayRunner`,
`InboundEvent`, `SessionRouter`, `build_session_key`.

**Next:** see `02-platform-adapters.ipynb` for the
`BasePlatformAdapter` contract, real Telegram/Slack adapters,
`SubprocessExecutor` (federal tier), and how `StreamBridge` consumes
`Delta`s and edits a single platform message in place rather than
spamming the channel.